In [ ]:
# [목적] 대화형 프롬프트, JSON 변환 도구, AI 모델을 준비합니다.
# JsonOutputParser를 쓰면 AI 답변을 코드에서 다루기 쉬운 JSON으로 받을 수 있습니다.
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

model = ChatOpenAI(temperature=0, model_name="gpt-5-nano")

In [ ]:
# [목적] AI가 반환할 결과의 모양을 description과 hashtags로 정합니다.
# Field의 설명은 AI가 각 항목에 무엇을 넣어야 하는지 이해하는 데 도움을 줍니다.
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [ ]:
# [목적] AI에게 물을 질문과 JSON 응답을 읽을 파서를 준비합니다.
# print는 AI에게 전달할 JSON 형식 안내문을 확인하는 학습용 코드입니다.
question = "지구 온난화의 심각성에 대해 알려주세요."

parser = JsonOutputParser(pydantic_object=Topic)
print(parser.get_format_instructions())

In [ ]:
# [목적] 시스템 역할, 질문, JSON 형식 안내를 묶어 AI 실행 흐름을 만듭니다.
# prompt | model | parser는 답변 생성 후 JSON으로 변환하는 순서를 뜻합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | model | parser

answer = chain.invoke({"question": question})

In [ ]:
# [목적] JSON 결과에서 설명(description) 항목만 꺼내 확인합니다.
# 같은 방식으로 answer["hashtags"]도 사용할 수 있습니다.
answer["description"]